# Row 3: the cue matcher alone (stage 0 and 1a)

What this tests and why: `docs/RATIONALE.md` (section 4, row 3; decisions D17 to D19). Run the cells in order from the repo root.

1. The edge model turns each question into 3 visual cues (stage 0).
2. The embedding model compares every 2 s window, every 0.5 s, with those cues and we store the similarities (stage 1a).
3. On the CPU we try different trigger rules and thresholds on the stored numbers, and score them.

The embedding model uses about 4.4 GB of GPU memory, the decider about 4.3 GB: **they do not fit together on the 4060, so the
two stages run one after the other, as separate commands** (the cells below do this).

## 0. Setup and GPU check

In [ ]:
import os
os.chdir(os.path.join(os.getcwd(), ".."))


In [2]:
%pip install -q -e ".[models]"
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, f"{p.total_memory/1e9:.1f} GB")

Note: you may need to restart the kernel to use updated packages.
torch 2.6.0+cu124 | cuda: True
NVIDIA GeForce RTX 4060 Laptop GPU 8.6 GB


ERROR: file:///C:/Users/abhin/Code/CV_Proj/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


## 1. Videos (skip if the val videos are already downloaded)

In [ ]:
!python scripts/ovo_tar_remote.py fetch --samples data/processed/ovo_far_val.jsonl --out data/raw/ovo/src_videos

## 2. Stage 0: the cues

The edge model reads each question and writes 3 short descriptions of what the camera would see at the right moment.
Takes a few minutes for the val questions.

In [3]:
!python scripts/run_cues.py --samples data/processed/ovo_far_val.jsonl

python: can't open file 'c:\\Users\\abhin\\Code\\CV_Proj\\notebooks\\scripts\\run_cues.py': [Errno 2] No such file or directory


## 2b. Read the cues

Skim these: if the cues are vague or wrong, the matcher cannot work well, and that is worth knowing before waiting for embeddings.

In [ ]:
import glob, json
f = sorted(glob.glob("results/cache/cues/*/cues.jsonl"))[-1]
rows = [json.loads(l) for l in open(f) if l.strip()]
print(f, len(rows), "questions")
for r in rows[:12]:
    print(f"\n{r['id']}: {r['query']}")
    for c in r["cues"]: print("   -", c)

## 3. Stage 1a smoke test: two of the shortest val questions

In [ ]:
import glob
CUES = sorted(glob.glob("results/cache/cues/*/cues.jsonl"))[-1]
!python scripts/run_embed_cache.py --samples data/processed/ovo_far_val.jsonl --cues {CUES} --limit 2 --shortest

## 4. Speed check

Median milliseconds per matcher step, and what that means for the whole val split (about 7,900 steps).

In [ ]:
import glob, json, statistics
files = glob.glob("results/cache/matcher/*/ovo_*.json")
ms = [json.load(open(f))["ms_median"] for f in files if json.load(open(f)).get("ms_median")]
print(f"{len(files)} questions stored; median {statistics.median(ms):.0f} ms per step")
print(f"projected for val (about 7,900 steps): {statistics.median(ms)*7900/60000:.0f} minutes")

## 5. Stage 1a on all val questions (resumable: if it stops, run this cell again)

In [ ]:
!python scripts/run_embed_cache.py --samples data/processed/ovo_far_val.jsonl --cues {CUES}

## 6. Score the matcher alone

Prints the Gate 2 diagnostic (does the similarity separate moments near an event from the rest?), then a table of
average F1 for each trigger rule, cue combination, threshold and duplicate handling.

In [ ]:
import glob
cache = sorted(glob.glob("results/cache/matcher/Ops-MM-embedding-v1-2B__*"))[-1]
!python scripts/replay_matcher.py --cache {cache} --samples data/processed/ovo_far_val.jsonl

## 7. Push the results

The `.json` files (similarities) and the cues are small and get pushed; the `.npy` window vectors are large, are kept
local on purpose and are ignored by git. In a terminal:
```
git add results/cache && git commit -m "Row 3 cues and matcher similarities, val" && git pull --rebase origin abhi && git push origin abhi
```